# Tax AI Colab GPU Backend — Gemma 4 E4B + api-ocr-2025

架構：**Gemma 4 E4B Vision sidecar (:8001) → api-ocr-2025 (:8080) → Cloudflare Temporary HTTPS Tunnel**

固定原則：買受人8格=buyer_tax_id；右下專用章=seller_tax_id；檢查碼只驗證不改值；看不清回 null。

api-ocr-2025 pinned: `5ef5794c1b0c3fc640d6ac8c8d26562b6c035202`

In [2]:
import subprocess, sys, platform
print("Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [1]:
!apt-get -qq update
!apt-get -qq install -y libzbar0
!pip -q install -U "transformers>=5.5.0" accelerate bitsandbytes huggingface_hub fastapi uvicorn python-multipart pillow requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package libfftw3-double3:amd64.
Preparing to unpack .../01-libfftw3-double3_3.3.8-2ubuntu8_amd64.deb ...
Unpacking libfftw3-double3:amd64 (3.3.8-2ubuntu8) ...
Selecting previously unselected package liblqr-1-0:amd64.
Preparing to unpack .../02-liblqr-1-0_0.4.2-2.1_amd64.deb ...
Unpacking liblqr-1-0:amd64 (0.4.2-2.1) ...
Selecting previously unselected package imagemagick-6-common.
Preparing to unpack .../03-imagemagick-6-common_8%3a6.9.11.60+dfsg-1.3ubuntu0.22.04.5_all.deb ...
Unpacking imagemagick

## Hugging Face 登入
若 Gemma 4 需要授權，先接受模型條款，再執行下一格。

In [ ]:
from huggingface_hub import login
login()

In [1]:
import os, subprocess, shutil
REPO="/content/api-ocr-2025"
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","https://github.com/adi-gov-tw/api-ocr-2025.git",REPO],check=True)
subprocess.run(["git","checkout","5ef5794c1b0c3fc640d6ac8c8d26562b6c035202"],cwd=REPO,check=True)

CompletedProcess(args=['git', 'checkout', '5ef5794c1b0c3fc640d6ac8c8d26562b6c035202'], returncode=0)

In [ ]:
!pip -q install -r /content/api-ocr-2025/requirements.txt

In [ ]:
import torch, gc
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
MODEL_ID="google/gemma-4-E4B-it"
processor=AutoProcessor.from_pretrained(MODEL_ID)
try:
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID,quantization_config=q,device_map="auto",dtype=torch.float16)
    print("Gemma 4 E4B loaded in 4-bit")
except Exception as e:
    print("4-bit failed, fallback:",e)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model=AutoModelForMultimodalLM.from_pretrained(MODEL_ID,device_map="auto",dtype="auto")
    print("Gemma 4 E4B loaded")

In [ ]:
sidecar_code='import base64, io, json, re, time\nfrom fastapi import FastAPI, Request\nfrom PIL import Image\nimport torch\napp=FastAPI()\n\ndef image_part(url):\n    if isinstance(url,dict): url=url.get("url")\n    if url and url.startswith("data:"):\n        return {"type":"image","base64":url.split(",",1)[1]}\n    return {"type":"image","url":url}\n\ndef collect(messages):\n    imgs=[]; texts=[]\n    for m in messages or []:\n        c=m.get("content")\n        if isinstance(c,str):\n            texts.append(c)\n        elif isinstance(c,list):\n            for p in c:\n                t=p.get("type")\n                if t in ("text","input_text"):\n                    texts.append(p.get("text",""))\n                elif t in ("image_url","input_image"):\n                    u=p.get("image_url") or p.get("image")\n                    if u: imgs.append(image_part(u))\n    return imgs,"\\n".join(texts)\n\n@app.get("/health")\nasync def health():\n    return {"status":"ok","model":"google/gemma-4-E4B-it","cuda":torch.cuda.is_available()}\n\n@app.get("/v1/models")\nasync def models():\n    return {"object":"list","data":[{"id":"google/gemma-4-E4B-it","object":"model"}]}\n\n@app.post("/v1/chat/completions")\nasync def chat(req:Request):\n    body=await req.json()\n    imgs,text=collect(body.get("messages",[]))\n    guard=("You are a Taiwan unified-invoice vision extraction model. "\n           "For triplicate/manual invoices, the 8 boxes after/below 買受人/統一編號 are buyer_tax_id; "\n           "the lower-right 統一發票專用章 contains seller_tax_id. Never swap them. "\n           "Never use checksum to invent or correct digits. If unreadable return null. "\n           "Return only the requested JSON or answer.")\n    if imgs:\n        messages=[{"role":"user","content":[imgs[0],{"type":"text","text":guard+"\\n"+text}]}]\n    else:\n        messages=[{"role":"user","content":text or "Reply OK"}]\n    inputs=processor.apply_chat_template(messages,tokenize=True,return_dict=True,return_tensors="pt",add_generation_prompt=True,enable_thinking=False).to(model.device)\n    n=inputs["input_ids"].shape[-1]\n    with torch.inference_mode():\n        out=model.generate(**inputs,max_new_tokens=min(int(body.get("max_tokens") or 900),1200),do_sample=False)\n    raw=processor.decode(out[0][n:],skip_special_tokens=False)\n    try:\n        parsed=processor.parse_response(raw,prefix=inputs["input_ids"])\n        content=parsed.get("content") if isinstance(parsed,dict) else str(parsed)\n    except Exception:\n        content=raw\n    content=re.sub(r"<\\|[^>]+\\|>","",content or "").strip()\n    return {"id":"chatcmpl-gemma4e4b","object":"chat.completion","created":int(time.time()),"model":"google/gemma-4-E4B-it","choices":[{"index":0,"message":{"role":"assistant","content":content},"finish_reason":"stop"}]}\n'
open("/content/gemma_e4b_sidecar.py","w",encoding="utf-8").write(sidecar_code)
print("sidecar ready")

In [4]:
import subprocess, time, requests
sidecar_proc=subprocess.Popen(["uvicorn","gemma_e4b_sidecar:app","--host","0.0.0.0","--port","8001"],cwd="/content")
for _ in range(60):
    try:
        r=requests.get("http://127.0.0.1:8001/health",timeout=2)
        if r.ok:
            print(r.json()); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("sidecar failed")

RuntimeError: sidecar failed

In [ ]:
import os, subprocess, time, requests
env=os.environ.copy()
env.update({
 "APIOCR_VLM_BACKEND":"local",
 "APIOCR_LOCAL_VLM_ENABLED":"true",
 "APIOCR_LOCAL_VLM_URL":"http://127.0.0.1:8001/v1",
 "APIOCR_LOCAL_VLM_MODEL":"google/gemma-4-E4B-it",
 "APIOCR_LOCAL_VLM_TIMEOUT":"180",
 "APIOCR_VLM_ENABLED":"true",
 "APIOCR_USE_GPU":"false",
 "APIOCR_DESKEW":"true",
 "APIOCR_UPSCALE_MIN_SIDE":"1400",
 "APIOCR_MIN_CONFIDENCE":"0.20"
})
api_proc=subprocess.Popen(["uvicorn","app.main:app","--host","0.0.0.0","--port","8080","--workers","1"],cwd="/content/api-ocr-2025",env=env)
for _ in range(90):
    try:
        r=requests.get("http://127.0.0.1:8080/health",timeout=2)
        if r.ok:
            print(r.json()); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("api-ocr failed")

In [ ]:
import requests, json
r=requests.get("http://127.0.0.1:8080/health/vlm",timeout=180)
print(r.status_code)
print(json.dumps(r.json(),ensure_ascii=False,indent=2))

## 建立臨時 HTTPS Tunnel
不需 Cloudflare 帳號。每次 Colab 重啟 URL 都會改變。把輸出的 `https://...trycloudflare.com` 貼回 Tax AI V1。

In [ ]:
import os, subprocess, re, time, requests
cf="/content/cloudflared"
if not os.path.exists(cf):
    subprocess.run(["wget","-q","https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64","-O",cf],check=True)
    os.chmod(cf,0o755)
tunnel_proc=subprocess.Popen([cf,"tunnel","--url","http://127.0.0.1:8080","--no-autoupdate"],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
public_url=None
deadline=time.time()+60
while time.time()<deadline:
    line=tunnel_proc.stdout.readline()
    if line:
        print(line.rstrip())
        m=re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",line)
        if m:
            public_url=m.group(0); break
if not public_url: raise RuntimeError("Tunnel URL not found")
print("\\nCOLAB_BACKEND_URL =",public_url)
print(requests.get(public_url+"/health",timeout=30).json())

In [ ]:
from google.colab import files
import requests, json
uploaded=files.upload()
for name,data in uploaded.items():
    r=requests.post(public_url+"/v1/invoice",files={"file":(name,data,"image/jpeg")},data={"engine":"vlm","slim":"false","include_image":"false"},timeout=240)
    print("HTTP",r.status_code)
    print(json.dumps(r.json(),ensure_ascii=False,indent=2))

## Regression Cases
僅用來驗證，不能硬編碼：
- XV25099553 → buyer `54169882`, seller `80113152`
- QU08904069 → buyer `22644758`, seller `27601907`